# Data Generation - CRM & ERP Synthetic Datasets

**Business context**

A company sells through two separate systems that don't always agree with each other. The sales team works in a CRM, negotiating and closing deals. Finance and logistics work in an ERP, the system that actually issues invoices and ships goods. In real companies these two systems are rarely perfectly in sync - different teams own them, and the integration between them is often manual, delayed, or broken. The gap between "what sales says was sold" and "what finance actually invoiced" is a classic reconciliation problem in larger organizations, and it's exactly what this project is built to detect.

This notebook generates two synthetic source datasets for the reconciliation pipeline:

- `data/raw/erp_orders_export.csv` - the ERP side: what was actually invoiced and shipped, a simulated nightly export.
- `data/raw/crm_deals_seed.json` - the CRM side: what the sales team recorded, seed data served later by the mock CRM API.

The two datasets are generated with intentional, known discrepancies between them (see the legend below), so the reconciliation logic built later in the project has real, verifiable cases to catch - not just clean data with nothing to find.

**Legend: reconciliation outcomes**

| Status | Meaning | Why it matters |
|---|---|---|
| `MATCHED` | Deal won in CRM, invoiced in ERP, amounts agree within 1% | The healthy case, nothing to act on |
| `AMOUNT_MISMATCH` | Invoiced, but for a different amount than the CRM deal | Could be a late discount, a partial shipment, or a pricing error - needs a human to check which system is right |
| `CRM_ONLY` | Deal marked Won in CRM, never invoiced in ERP | The serious case: either the deal fell through after being marked Won, or it's real revenue that was never billed - a direct financial loss |
| `ERP_ONLY` | An ERP order points to a CRM deal that doesn't exist at all | A data integrity error - a typo in the ID, a deleted CRM record - something structurally broken that needs investigating, not just booking |

Also present in the data but outside the reconciliation scope:

- **Organic orders** - ERP orders with no CRM link at all (`crm_deal_id` is empty), i.e. direct sales that never went through the CRM pipeline.
- **Data quality issues** - a handful of negative amounts, missing customer IDs, and duplicate rows, injected on purpose to give the data-validation logic something real to catch later.

---

*(Polski)*

**Kontekst biznesowy**

Firma sprzedaje przez dwa oddzielne systemy, które nie zawsze się ze sobą zgadzają. Dział sprzedaży pracuje w CRM, negocjuje i zamyka transakcje (deale). Dział finansów i logistyki pracuje w systemie ERP, czyli tym, który faktycznie wystawia faktury i wysyła towar. W prawdziwych firmach te dwa systemy rzadko są ze sobą idealnie zsynchronizowane - obsługują je różne zespoły, a integracja między nimi bywa ręczna, opóźniona albo wadliwa. Rozjazd między tym "co dział sprzedaży twierdzi że sprzedał" a tym "co finanse faktycznie zafakturowały" to klasyczny problem kontrolingu w większych organizacjach - i dokładnie to ten projekt ma za zadanie wykrywać.

Ten notebook generuje dwa syntetyczne zbiory danych źródłowych dla pipeline'u rekoncyliacji:

- `data/raw/erp_orders_export.csv` - strona ERP: co faktycznie zostało zafakturowane i wysłane, symulacja nocnego eksportu.
- `data/raw/crm_deals_seed.json` - strona CRM: co zarejestrował dział sprzedaży, dane początkowe serwowane później przez mockowe API CRM.

Oba zbiory danych są generowane z celowymi, znanymi z góry rozbieżnościami między sobą (patrz legenda poniżej), dzięki czemu logika rekoncyliacji budowana w dalszej części projektu ma na czym realnie się sprawdzić - a nie tylko czyste dane, w których nie ma niczego do znalezienia.

**Legenda: wyniki rekoncyliacji**

| Status | Znaczenie | Dlaczego to ważne |
|---|---|---|
| `MATCHED` | Deal wygrany w CRM, zafakturowany w ERP, kwoty zgodne w granicach 1% | Stan pożądany, nic do zrobienia |
| `AMOUNT_MISMATCH` | Zafakturowano, ale za inną kwotę niż w dealu CRM | Może to być późniejszy rabat, częściowa wysyłka albo błąd cenowy - wymaga sprawdzenia przez człowieka, który system ma rację |
| `CRM_ONLY` | Deal oznaczony jako wygrany w CRM, nigdy niezafakturowany w ERP | Najpoważniejszy przypadek: albo deal upadł już po oznaczeniu jako wygrany, albo to realny przychód, który nigdy nie został zafakturowany - bezpośrednia strata finansowa |
| `ERP_ONLY` | Zamówienie w ERP wskazuje na deal CRM, który w ogóle nie istnieje | Błąd integralności danych - literówka w ID, usunięty rekord CRM - coś nie zgadza się strukturalnie i wymaga wyjaśnienia, nie tylko zaksięgowania |

Dodatkowo w danych, ale poza zakresem rekoncyliacji:

- **Zamówienia organiczne** - zamówienia ERP bez żadnego powiązania z CRM (`crm_deal_id` puste), czyli sprzedaż bezpośrednia, która nigdy nie przechodziła przez CRM.
- **Usterki jakości danych** - kilka ujemnych kwot, brakujących identyfikatorów klienta i zduplikowanych wierszy, wstrzyknięte celowo, aby logika walidacji danych miała później na czym się sprawdzić.

In [ ]:
# Imports & Configuration

import json
import os
import random
from datetime import datetime, timedelta

import pandas as pd
from faker import Faker

random.seed(42)
fake = Faker()
Faker.seed(42)

OUTPUT_DIR = "../data/raw"
ERP_OUTPUT_PATH = f"{OUTPUT_DIR}/erp_orders_export.csv"
CRM_OUTPUT_PATH = f"{OUTPUT_DIR}/crm_deals_seed.json"
os.makedirs(OUTPUT_DIR, exist_ok=True)

DATE_START = datetime(2025, 1, 1)
DATE_END = datetime(2025, 12, 31)

In [3]:
# Reference Data

COUNTRIES = ["PL", "DE", "FR", "ES", "IT"]
CURRENCIES = ["PLN", "EUR", "USD"]
SALES_REPS = [fake.name() for _ in range(8)]
SKUS = [f"SKU-{i:04d}" for i in range(1, 31)]
STAGES = ["Won", "Open", "Lost"]
STAGE_WEIGHTS = [0.65, 0.20, 0.15]


def random_date(start, end):
    delta = end - start
    return start + timedelta(days=random.randint(0, delta.days))

## Step 1 - CRM Deals

Generates 200 CRM deals - the sales team's version of events.

Each deal has a **stage**: `Won` (deal closed successfully), `Open` (still being negotiated), or `Lost` (fell through). Only `Won` deals matter for the reconciliation logic later, since only a won deal should be invoiced. `Open` and `Lost` deals exist here just to make the dataset look like a real CRM export, not to be reconciled against anything.

Stage mix: roughly 65% Won, 20% Open, 15% Lost.

---

*(Polski)*

## Krok 1 - Deale CRM

Generuje 200 dealów CRM, czyli wersję wydarzeń z perspektywy działu sprzedaży.

Każdy deal ma **etap**: `Won` (transakcja zamknięta pomyślnie), `Open` (wciąż negocjowana) albo `Lost` (nieudana). Tylko deale `Won` mają znaczenie dla logiki rekoncyliacji w dalszej części, bo tylko wygrany deal powinien zostać zafakturowany. `Open` i `Lost` istnieją tutaj tylko po to, żeby zbiór danych wyglądał jak prawdziwy eksport z CRM - nie są z niczym porównywane.

Rozkład etapów: mniej więcej 65% Won, 20% Open, 15% Lost.

In [4]:
# Generate CRM Deals (seed for the mock API)

N_DEALS = 200

crm_deals = []
for i in range(1, N_DEALS + 1):
    deal_id = f"DEAL-{i:05d}"
    stage = random.choices(STAGES, weights=STAGE_WEIGHTS)[0]
    close_date = random_date(DATE_START, DATE_END)
    currency = random.choice(CURRENCIES)
    amount = round(random.uniform(500, 25000), 2)

    crm_deals.append({
        "deal_id": deal_id,
        "account_id": f"ACC-{random.randint(1, 120):04d}",
        "amount": amount,
        "currency": currency,
        "close_date": close_date.strftime("%Y-%m-%d"),
        "sales_rep": random.choice(SALES_REPS),
        "country_code": random.choice(COUNTRIES),
        "stage": stage,
    })

crm_df = pd.DataFrame(crm_deals)
crm_df["stage"].value_counts()

stage
Won     131
Lost     35
Open     34
Name: count, dtype: int64

## Step 2 - ERP Orders Linked to Won Deals

This is where the three main reconciliation statuses are created on purpose. Every `Won` CRM deal is routed into one of three buckets:

- **MATCHED** (~70% of Won deals) - an ERP order is created for the deal, amount within 1% of the CRM amount. This is what a healthy, well-integrated system looks like.
- **AMOUNT_MISMATCH** (~15% of Won deals) - an ERP order is created, but the amount is deliberately off by 5-25%. In a real company this might be a late-applied discount, a partial shipment, or a manual entry error.
- **CRM_ONLY** (the remainder, ~15% of Won deals) - no ERP order is created at all. In business terms, this is a deal the sales team marked as won that was never actually invoiced - potential lost revenue, and the case a control/risk team would care about most.

---

*(Polski)*

## Krok 2 - Zamówienia ERP powiązane z wygranymi dealami

To tutaj celowo powstają trzy główne statusy rekoncyliacji. Każdy wygrany (`Won`) deal CRM trafia do jednego z trzech koszyków:

- **MATCHED** (~70% wygranych deali) - powstaje zamówienie ERP dla deala, kwota zgodna w granicach 1% z kwotą z CRM. Tak wygląda zdrowy, dobrze zintegrowany system.
- **AMOUNT_MISMATCH** (~15% wygranych deali) - powstaje zamówienie ERP, ale kwota jest celowo zaniżona lub zawyżona o 5-25%. W prawdziwej firmie mógłby to być późno naliczony rabat, częściowa wysyłka albo błąd przy ręcznym wpisywaniu.
- **CRM_ONLY** (pozostałe ~15% wygranych deali) - żadne zamówienie ERP w ogóle nie powstaje. W ujęciu biznesowym to deal, który dział sprzedaży oznaczył jako wygrany, ale nigdy faktycznie nie został zafakturowany - potencjalna utrata przychodu i przypadek, na którym najbardziej zależałoby zespołowi ds. kontroli i ryzyka.

In [ ]:
# Generate ERP Orders Linked to Won CRM Deals

won_deals = crm_df[crm_df["stage"] == "Won"].to_dict("records")
random.shuffle(won_deals)

n_won = len(won_deals)
n_matched = int(n_won * 0.70)
n_mismatch = int(n_won * 0.15)
# remainder -> CRM_ONLY, no ERP order generated

matched_deals = won_deals[:n_matched]
mismatch_deals = won_deals[n_matched:n_matched + n_mismatch]
crm_only_deals = won_deals[n_matched + n_mismatch:]

erp_orders = []
order_counter = 1


def make_erp_order(order_id, crm_deal_id, deal, amount_override=None):
    quantity = random.randint(1, 20)
    order_date = datetime.strptime(deal["close_date"], "%Y-%m-%d") + timedelta(days=random.randint(0, 3))
    invoice_date = order_date + timedelta(days=random.randint(0, 5))
    amount = amount_override if amount_override is not None else deal["amount"]
    unit_price = round(amount / quantity, 2)

    return {
        "order_id": order_id,
        "crm_deal_id": crm_deal_id,
        "customer_id": f"CUST-{random.randint(1, 300):04d}",
        "sku": random.choice(SKUS),
        "quantity": quantity,
        "unit_price": unit_price,
        "currency": deal["currency"],
        "order_date": order_date.strftime("%Y-%m-%d"),
        "invoice_date": invoice_date.strftime("%Y-%m-%d"),
        "country_code": deal["country_code"],
        "amount": round(unit_price * quantity, 2),
    }


# MATCHED: ERP amount within 1% of CRM amount
for deal in matched_deals:
    order_id = f"ORD-{order_counter:05d}"
    order_counter += 1
    variance = random.uniform(-0.01, 0.01)
    amount = round(deal["amount"] * (1 + variance), 2)
    erp_orders.append(make_erp_order(order_id, deal["deal_id"], deal, amount_override=amount))


# AMOUNT_MISMATCH: ERP amount differs by 5-25%, random direction
for deal in mismatch_deals:
    order_id = f"ORD-{order_counter:05d}"
    order_counter += 1
    variance = random.choice([-1, 1]) * random.uniform(0.05, 0.25)
    amount = round(deal["amount"] * (1 + variance), 2)
    erp_orders.append(make_erp_order(order_id, deal["deal_id"], deal, amount_override=amount))


print(f"MATCHED deals:         {len(matched_deals)}")
print(f"AMOUNT_MISMATCH deals: {len(mismatch_deals)}")
print(f"CRM_ONLY deals (no ERP order): {len(crm_only_deals)}")

MATCHED deals:         91
AMOUNT_MISMATCH deals: 19
CRM_ONLY deals (no ERP order): 21


## Step 3 - ERP-Only Anomalies and Organic Orders

Two more categories of ERP orders, generated independently of the CRM deals above:

- **ERP_ONLY** - 12 orders that reference a `crm_deal_id` which doesn't exist anywhere in the CRM data at all. This simulates a broken reference: a typo during manual entry, a CRM record deleted after the ERP order referenced it, or a sync that partially failed. It's a data integrity problem, not a financial one - it needs investigating rather than booking.
- **Organic orders** - 150 orders with no `crm_deal_id` at all (`NULL`). These represent direct sales that never went through the CRM pipeline in the first place - for example walk-in customers or orders placed outside the sales process. They're intentionally outside the reconciliation scope: there's nothing on the CRM side to compare them against, so the pipeline should leave them alone rather than flag them as a mismatch.

---

*(Polski)*

## Krok 3 - Anomalie ERP-only i zamówienia organiczne

Dwie kolejne kategorie zamówień ERP, generowane niezależnie od deali CRM opisanych wyżej:

- **ERP_ONLY** - 12 zamówień odwołujących się do `crm_deal_id`, który w ogóle nie istnieje w danych CRM. Symuluje to zerwane powiązanie: literówkę przy ręcznym wpisywaniu, rekord CRM usunięty już po tym jak zamówienie ERP się do niego odwołało, albo częściowo nieudaną synchronizację. To problem integralności danych, nie finansowy - wymaga wyjaśnienia, nie księgowania.
- **Zamówienia organiczne** - 150 zamówień bez żadnego `crm_deal_id` (`NULL`). Reprezentują sprzedaż bezpośrednią, która nigdy nie przechodziła przez CRM, na przykład klientów obsłużonych bezpośrednio albo zamówienia złożone poza standardowym procesem sprzedaży. Są celowo poza zakresem rekoncyliacji: po stronie CRM nie ma z czym ich porównać, więc pipeline powinien je zostawić w spokoju, a nie oznaczać jako niezgodność.

In [ ]:
# Generate ERP-Only Anomalies and Organic Orders

N_ERP_ONLY = 12
N_ORGANIC = 150


# ERP_ONLY: crm_deal_id set to an id that does not exist in crm_df
for _ in range(N_ERP_ONLY):
    order_id = f"ORD-{order_counter:05d}"
    order_counter += 1
    fake_deal_id = f"DEAL-{random.randint(90000, 99999):05d}"  # outside the real 1-200 id range
    quantity = random.randint(1, 20)
    unit_price = round(random.uniform(50, 1500), 2)
    order_date = random_date(DATE_START, DATE_END)
    invoice_date = order_date + timedelta(days=random.randint(0, 5))

    erp_orders.append({
        "order_id": order_id,
        "crm_deal_id": fake_deal_id,
        "customer_id": f"CUST-{random.randint(1, 300):04d}",
        "sku": random.choice(SKUS),
        "quantity": quantity,
        "unit_price": unit_price,
        "currency": random.choice(CURRENCIES),
        "order_date": order_date.strftime("%Y-%m-%d"),
        "invoice_date": invoice_date.strftime("%Y-%m-%d"),
        "country_code": random.choice(COUNTRIES),
        "amount": round(unit_price * quantity, 2),
    })


# Organic orders: no CRM link at all
for _ in range(N_ORGANIC):
    order_id = f"ORD-{order_counter:05d}"
    order_counter += 1
    quantity = random.randint(1, 20)
    unit_price = round(random.uniform(50, 1500), 2)
    order_date = random_date(DATE_START, DATE_END)
    invoice_date = order_date + timedelta(days=random.randint(0, 5))

    erp_orders.append({
        "order_id": order_id,
        "crm_deal_id": None,
        "customer_id": f"CUST-{random.randint(1, 300):04d}",
        "sku": random.choice(SKUS),
        "quantity": quantity,
        "unit_price": unit_price,
        "currency": random.choice(CURRENCIES),
        "order_date": order_date.strftime("%Y-%m-%d"),
        "invoice_date": invoice_date.strftime("%Y-%m-%d"),
        "country_code": random.choice(COUNTRIES),
        "amount": round(unit_price * quantity, 2),
    })

erp_df = pd.DataFrame(erp_orders)
len(erp_df)

272

## Step 4 - Inject Data Quality Issues

On top of the reconciliation scenarios above, a few ordinary data quality problems are added - the kind that show up in almost any real data export, regardless of whether the numbers reconcile:

- **3 negative amounts** - a data entry error, an order that should never have a negative value.
- **3 missing `customer_id` values** - an incomplete record.
- **2 duplicate rows** - the same order appearing twice, as if the export ran twice or a sync glitched.

These aren't part of the CRM/ERP reconciliation story - they exist to give the data-validation step of the pipeline (built later, in `transform.py`) something concrete to detect and clean before the reconciliation logic even runs.

---

*(Polski)*

## Krok 4 - Wstrzyknięcie usterek jakości danych

Poza scenariuszami rekoncyliacji opisanymi wyżej, dodano też kilka zwykłych problemów jakości danych - takich, jakie pojawiają się w niemal każdym prawdziwym eksporcie danych, niezależnie od tego, czy liczby się zgadzają:

- **3 ujemne kwoty** - błąd przy wpisywaniu danych, zamówienie nigdy nie powinno mieć ujemnej wartości.
- **3 brakujące wartości `customer_id`** - niekompletny rekord.
- **2 zduplikowane wiersze** - to samo zamówienie pojawia się dwukrotnie, jakby eksport uruchomiono dwa razy albo synchronizacja się zacięła.

To nie jest część historii rekoncyliacji CRM-ERP - istnieją po to, żeby etap walidacji danych w pipeline (budowany później, w `transform.py`) miał na czym realnie się sprawdzić, zanim w ogóle uruchomi się logika rekoncyliacji.

In [ ]:
# Inject Data Quality Issues

erp_df = erp_df.sample(frac=1, random_state=42).reset_index(drop=True)

# Negative amounts (data entry error)
neg_idx = erp_df.sample(n=3, random_state=1).index
erp_df.loc[neg_idx, "amount"] = -erp_df.loc[neg_idx, "amount"].abs()


# NULL customer_id (incomplete record)
null_idx = erp_df.sample(n=3, random_state=2).index
erp_df.loc[null_idx, "customer_id"] = None


# Duplicate rows (simulates a double-export glitch)
dup_rows = erp_df.sample(n=2, random_state=3).copy()
erp_df = pd.concat([erp_df, dup_rows], ignore_index=True)

erp_df = erp_df.sample(frac=1, random_state=7).reset_index(drop=True)
len(erp_df)

274

## Step 5 - Export

Writes both datasets to disk. The CSV becomes the file `extract.py` reads directly, standing in for a nightly ERP export. The JSON becomes the seed data behind the mock CRM API built in `crm_api.py` - which is why it's exported as JSON rather than CSV, it needs to look like something a real API would return.

---

*(Polski)*

## Krok 5 - Eksport

Zapisuje oba zbiory danych na dysk. CSV staje się plikiem, który `extract.py` odczytuje bezpośrednio, symulując nocny eksport z ERP. JSON staje się danymi źródłowymi dla mockowego API CRM budowanego w `crm_api.py` - dlatego jest eksportowany jako JSON a nie CSV, musi wyglądać jak coś, co zwróciłoby prawdziwe API.

In [ ]:
# Export

erp_df.to_csv(ERP_OUTPUT_PATH, index=False)

with open(CRM_OUTPUT_PATH, "w", encoding="utf-8") as f:
    json.dump(crm_deals, f, indent=2, ensure_ascii=False)

print(f"ERP orders exported: {len(erp_df)} rows -> {ERP_OUTPUT_PATH}")
print(f"CRM deals exported:  {len(crm_deals)} rows -> {CRM_OUTPUT_PATH}")

ERP orders exported: 274 rows -> ../data/raw/erp_orders_export.csv
CRM deals exported:  200 rows -> ../data/raw/crm_deals_seed.json


## Ground Truth Summary

This is the "answer key." Because every discrepancy above was planted on purpose, we already know exactly how many orders should end up in each reconciliation bucket. It's printed here for reference only - it is deliberately **not** saved to any file the pipeline reads.

The point is that later, `transform.py` will work only from the raw CSV and JSON, with no access to this summary, and derive its own classification. If its output matches these numbers, that's proof the reconciliation logic actually works, not just that the code runs without errors.

---

*(Polski)*

## Podsumowanie danych referencyjnych (ground truth)

To jest "klucz odpowiedzi". Ponieważ każda rozbieżność opisana wyżej została wstrzyknięta celowo, już teraz wiemy dokładnie, ile zamówień powinno trafić do każdego koszyka rekoncyliacji. Wypisane tu wyłącznie dla wglądu, celowo **nie** jest zapisywane do żadnego pliku, z którego korzysta pipeline.

Chodzi o to, że późniejszy `transform.py` będzie pracować wyłącznie na surowych plikach CSV i JSON, bez dostępu do tego podsumowania, i samodzielnie wyprowadzi własną klasyfikację. Jeśli jego wynik zgadza się z tymi liczbami, to dowód, że logika rekoncyliacji faktycznie działa, a nie tylko że kod się uruchamia bez błędów.

In [9]:
# Ground Truth Summary (not exported)

print("Expected reconciliation outcome once the pipeline runs:")
print(f"  MATCHED         : {len(matched_deals)}")
print(f"  AMOUNT_MISMATCH : {len(mismatch_deals)}")
print(f"  CRM_ONLY        : {len(crm_only_deals)}")
print(f"  ERP_ONLY        : {N_ERP_ONLY}")
print(f"  organic orders (outside reconciliation scope): {N_ORGANIC}")
print("  injected data quality issues: 3 negative amounts, 3 NULL customer_id, 2 duplicate rows")

Expected reconciliation outcome once the pipeline runs:
  MATCHED         : 91
  AMOUNT_MISMATCH : 19
  CRM_ONLY        : 21
  ERP_ONLY        : 12
  organic orders (outside reconciliation scope): 150
  injected data quality issues: 3 negative amounts, 3 NULL customer_id, 2 duplicate rows
